## Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Answer:

Traditional MapReduce has several limitations. It writes intermediate results to disk, which makes processing slower, especially for iterative tasks. It also requires more complex code and is not efficient for real-time or interactive processing.

Spark is preferred because it supports in-memory processing, reduces disk I/O, and provides APIs for SQL, streaming, and machine learning.

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [5]:
spark = SparkSession.builder.appName("Week5 Assignment").getOrCreate()

In [6]:
df = spark.read.csv(
    r"C:\Week5_PySpark_Assignment\data\Sample - Superstore.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"'
)

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [10]:
df.show(5)
df.printSchema()
df.columns
df.count()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

9994

## Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Answer:

Spark stores intermediate data in memory (RAM) instead of writing it to disk after every operation. This reduces disk I/O, making iterative machine learning algorithms much faster because the same data can be reused directly from memory.

In [7]:
# Q3
# Dataset Change:
# Assignment uses 'user_id' and 'transaction_date'.
# Superstore dataset uses 'Order ID' and 'Product ID' as the closest equivalent.

df_q3 = df.dropDuplicates(["Order ID", "Product ID"])

print("Original Rows:", df.count())
print("After Removing Duplicates:", df_q3.count())

df_q3.show(5)

Original Rows: 9994
After Removing Duplicates: 9986
+------+--------------+----------+---------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|   Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+---------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|  6288|CA-2014-100090|  7/8/2014|7/12/2014|Standard Class|   EB-13705|      Ed Braxton|  Corporate|United States|San Francisco|California|      94122|  West|FUR-TA-100

In [8]:
# Q4: Filter West region and calculate average sales

df_q4 = (
    df
    .filter(col("Region") == "West")
    .groupBy("Category")
    .agg(avg("Sales").alias("average_sale_amount"))
)

df_q4.show()

+---------------+-------------------+
|       Category|average_sale_amount|
+---------------+-------------------+
|Office Supplies|   116.422376910912|
|      Furniture|   357.302324611033|
|     Technology|   420.687532554257|
+---------------+-------------------+



In [9]:
# Q5
# Dataset Change:
# Assignment uses 'status' column.
# Superstore dataset uses 'Ship Mode' because 'status' is not available.
#.na.drop() removes rows that contain null values, while .na.fill() replaces null values with a specified value.
#For example, null values in the status column can be replaced with "Unknown".

df_q5 = df.na.fill({"Ship Mode": "Unknown"})

df_q5.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [44]:
# Q6
# Dataset Change:
# Assignment uses 'city'.
# Superstore dataset already contains 'City', so no changes are required.

from pyspark.sql.functions import count

df_q6 = df.groupBy("City") \
          .agg(count("*").alias("Total_Records")) \
          .filter("Total_Records > 100")

df_q6.show()

+-------------+-------------+
|         City|Total_Records|
+-------------+-------------+
|  Springfield|          163|
|       Dallas|          157|
| Philadelphia|          537|
|  Los Angeles|          747|
|San Francisco|          510|
|    San Diego|          170|
|      Detroit|          115|
|     Columbus|          222|
|      Chicago|          314|
|      Seattle|          428|
|New York City|          915|
|      Houston|          377|
| Jacksonville|          125|
+-------------+-------------+



## Q7. How does the immutability of Spark DataFrames affect how you perform data cleaning steps like dropping columns or renaming them?

Answer:

Spark DataFrames are immutable, which means they cannot be modified directly. Whenever we perform operations like dropping columns, renaming columns, or filtering data, Spark creates a new DataFrame while keeping the original DataFrame unchanged.

In [45]:
# Q8
# Dataset Change:
# Assignment uses 'age' and 'subscription'.
# Superstore dataset uses 'Quantity' and 'Segment'
# because 'age' and 'subscription' are not available.

df_q8 = df.filter(
    (df["Quantity"].between(2, 5)) &
    (df["Segment"] == "Consumer")
)

df_q8.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+--------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name| Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+--------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute|Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset Col.

## Q9. When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

Answer:

Handling null values before performing aggregation functions like `sum()` or `avg()` improves data quality and ensures accurate results. Missing values may lead to incorrect or incomplete calculations if they are not handled properly.

In [10]:
# Q10
# Dataset Change:
# Assignment uses 'raw_timestamp'.
# Superstore dataset uses 'Order Date'
# as the available date column.

from pyspark.sql.functions import col, to_timestamp

df_q10 = (
    df
    .withColumn(
        "Order Date",
        to_timestamp(col("Order Date"), "M/d/yyyy")
    )
    .withColumnRenamed("Order Date", "event_time")
)

df_q10.show(5)

+------+--------------+-------------------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|         event_time| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+-------------------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08 00:00:00|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      F

## Q11. Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

Answer:

Shuffle is the process of redistributing data across different partitions during operations like `groupBy()`, `join()`, or `orderBy()`. It is called a wide transformation because data moves between multiple partitions, increasing network communication and processing time.

In [38]:
# Q12
# Dataset Change:
# Assignment uses 'email' and 'username'.
# Superstore dataset doesn't have these columns,
# so 'Customer Name' and 'City' are used instead.

from pyspark.sql.functions import col, trim

df_q12 = df.filter(
    col("Customer Name").isNotNull() &
    (trim(col("City")) != "")
)

df_q12.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [39]:
# Q13
# Dataset Change:
# Assignment asks for 'price'.
# Superstore dataset uses 'Sales' as the equivalent numeric column.

from pyspark.sql.functions import min, max, mean

df_q13 = df.agg(
    min("Sales").alias("Minimum_Sales"),
    max("Sales").alias("Maximum_Sales"),
    mean("Sales").alias("Average_Sales")
)

df_q13.show()

+-------------+-------------+-----------------+
|Minimum_Sales|Maximum_Sales|    Average_Sales|
+-------------+-------------+-----------------+
|        0.444|     22638.48|229.8580008304938|
+-------------+-------------+-----------------+



## Q14. In the context of cleaning a dataset, what is the risk of using `inferSchema=True` when your source data contains messy or inconsistent date formats?
Answer:

Using `inferSchema=True` with inconsistent or messy data may assign incorrect data types to columns. For example, a date column containing different date formats may be detected as a String instead of a Date, leading to errors during data cleaning and analysis.

In [40]:
# Q15
# Dataset Change:
# Assignment uses 'store_id' and 'price'.
# Superstore dataset doesn't have 'store_id',
# so 'Category' is used for grouping.
# 'Sales' is used instead of 'price'.

from pyspark.sql.functions import sum

df_q15 = (
    df.dropDuplicates(["Order ID", "Product ID"])
      .na.fill({"Sales": 0})
      .groupBy("Category")
      .agg(sum("Sales").alias("Total_Revenue"))
)

df_q15.show()

+---------------+-----------------+
|       Category|    Total_Revenue|
+---------------+-----------------+
|Office Supplies|718317.7920000013|
|      Furniture|741432.0432999989|
|     Technology|835759.7369999974|
+---------------+-----------------+

